[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeteoSwiss/opendata-nwp-demos/blob/main/07_where_will_it_rain_next_24h.ipynb)

# 🌧️ Where Will It Rain In The Next 24h?

Planning a hike or outdoor event? Knowing **where it might rain in the next 24h** can make all the difference — and a **[probabilistic forecast](https://www.meteoswiss.admin.ch/weather/weather-and-climate-from-a-to-z/probability-in-forecasts.html)** gives more insight than a simple yes-no answer.

This notebook uses the **ICON-CH1-EPS** ensemble system to map the probability that total precipitation will exceed **0.1 mm/day** over the next 24 hours across Switzerland and surrounding regions.


### 🔍 What This Notebook Covers:
- 📥 Fetch perturbed ICON-CH1-EPS forecast data using [meteodata-lab](https://meteoswiss.github.io/meteodata-lab/)'s `ogd_api` module
- 🗺️ Visualize precipitation probabilities with [earthkit-plots](https://earthkit-plots.readthedocs.io/en/latest/examples/guide/01-introduction.html)

Let’s explore the forecast and answer the question:  **Where will it rain in the next 24h?**

## 📦 Install Dependencies (Colab only)
This cell installs all required dependencies if you're running the notebook in Google Colab. It is automatically skipped when running in a local Jupyter environment.

In [ ]:
# 📦 Google Colab Setup (skipped if not running in Colab)
import sys

def is_colab():
    return "google.colab" in sys.modules

if is_colab():
    !git clone https://github.com/MeteoSwiss/opendata-nwp-demos.git
    %cd opendata-nwp-demos
    !pip install poetry && poetry config virtualenvs.in-project true && poetry install --no-ansi
    import sys, os, pathlib
    venv = pathlib.Path(".venv")
    site = venv / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
    sys.path.insert(0, str(site))
    os.environ["ECCODES_DEFINITION_PATH"] = str((venv / "share/eccodes-cosmo-resources/definitions").resolve())

## 📥 Retrieve ICON-CH1-EPS Data

In [ ]:
from meteodatalab import ogd_api
from datetime import timedelta


# Create request
req = ogd_api.Request(
    collection="ogd-forecasting-icon-ch1",
    variable="TOT_PREC",
    ref_time="latest",
    perturbed=True,           # ensemble mode
    lead_time=timedelta(hours=24)
)

# Fetch data
da = ogd_api.get_from_ogd(req)

### Regridding to Regular Grid (EPSG:4326)

Model output is remapped to a regular lat/lon grid using `iconremap()` from `meteodatalab.regrid` for visualization.

In [ ]:
from rasterio.crs import CRS
from meteodatalab.operators import regrid

# Define ~1 km target grid over ICON-CH1-EPS domain
extent = (-0.817, 18.183, 41.183, 51.183)  # (xmin, xmax, ymin, ymax)
nx, ny = 429, 295

# Create regular lat/lon grid (EPSG:4326) and regrid
destination = regrid.RegularGrid(CRS.from_epsg(4326), nx, ny, *extent)
da_geo = regrid.iconremap(da, destination)

## 🎯 Compute Probability > 0.1 mm

In [ ]:
import numpy as np

# Boolean mask where member values > 0.1 mm
mask = (da_geo > 0.1)
prob_prec = mask.mean(dim="eps") * 100
prob_prec.attrs = da.attrs
prob_prec.attrs["long_name"] = "Probability of precipitation > 0.1 mm"
prob_prec.attrs["units"] = "%"

## 🗺️ Plot Probability Map

In [ ]:
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
from earthkit.plots.geo import bounds, domains
from earthkit.plots.styles import Style, Contour
import earthkit


# === Setup Domain (Switzerland) ===
bbox = bounds.BoundingBox(5.7, 10.5, 45.6, 48, ccrs.Geodetic())
domain = domains.Domain.from_bbox(bbox=bbox)

# === Define Style ===
levels = [5, 35, 65, 95, 100.01]  # Probability thresholds

contourf_style = Style(
    levels=levels,
    colors="YlGnBu",
    legend_style="colorbar",
)


contour_style = Contour(
    levels=levels,
    legend_style=None
)

# === Create Map ===
chart = earthkit.plots.Map(domain=domain)

# Plot filled contours and line contours
chart.contourf(prob_prec, x="lon", y="lat", style=contourf_style)
chart.contour(prob_prec, x="lon", y="lat", style=contour_style)

# Add map features
chart.borders()
chart.cities("high")
chart.land()
chart.legend(label="Total precipitation probability >0.1 mm/day (%)")

# === Title Setup ===
# Extract reference and lead time
ref_time = pd.to_datetime(prob_prec.coords["ref_time"].values[0])
lead_time = prob_prec.coords["lead_time"].values[0]
lead_hours = int(lead_time.astype("timedelta64[h]") / np.timedelta64(1, "h"))
valid_time = ref_time + pd.to_timedelta(lead_hours, unit="h")

# Format and apply title
chart.title("Switzerland Rain Prob >0.1mm Next 24h")

# === Show Plot ===
chart.show()


In [ ]:
import copy

# Strip non-serializable attrs (earthkit WrappedMetadata) before saving to NetCDF

def clean_attrs(attrs):
    """Keep only NetCDF-serializable attribute values."""
    valid_types = (str, int, float, bytes, list, tuple)
    cleaned = {}
    for k, v in attrs.items():
        if isinstance(v, valid_types):
            cleaned[k] = v
        else:
            # Try converting to string as a fallback
            try:
                cleaned[k] = str(v)
            except Exception:
                pass  # Drop completely if it can't be converted
    return cleaned

# Apply cleaned attrs to a copy, then save
prob_prec_nc = prob_prec.copy()
prob_prec_nc.attrs = clean_attrs(prob_prec.attrs)

prob_prec_nc.to_netcdf('rain_probability_next24h.nc')
print("Saved successfully!")

In [ ]:
# Inspect what's inside the WrappedMetadata object
raw_meta = prob_prec.attrs.get('metadata')
if raw_meta is not None:
    for key in ['shortName', 'paramId', 'typeOfLevel', 'edition']:
        try:
            print(f"{key}: {raw_meta[key]}")
        except Exception:
            pass

In [ ]:
prob_prec_nc = prob_prec.copy()
prob_prec_nc.attrs = {
    'long_name': prob_prec.attrs.get('long_name', 'Probability of precipitation > 0.1 mm'),
    'units': prob_prec.attrs.get('units', '%'),
    # Add any other serializable fields here
}
prob_prec_nc.to_netcdf('rain_probability_next24h.nc')

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

ds = xr.open_dataset('rain_probability_next24h.nc')
da = ds['__xarray_dataarray_variable__']  # or the actual variable name

da.plot(x='lon', y='lat', cmap='YlGnBu', vmin=0, vmax=100)
plt.title('Rain Probability > 0.1 mm – Next 24h')
plt.show()


In [ ]:
pip install datashader

In [ ]:
pip install ncplot

In [ ]:
from ncplot import view
view('rain_probability_next24h.nc')

In [1]:
from meteodatalab import ogd_api
from datetime import timedelta


# Create request
req = ogd_api.Request(
    collection="ogd-forecasting-icon-ch1",
    variable="TOT_PREC",
    ref_time="latest",
    perturbed=True,           # ensemble mode
    lead_time=timedelta(hours=24)
)

# Fetch data
da = ogd_api.get_from_ogd(req)

Earthkit-data caching is recommended. See: https://earthkit-data.readthedocs.io/en/latest/examples/cache.html


icon-ch1-eps-202603310900-24-tot_prec-perturb.grib2?AWSAccessKeyId=13GC1T4NT2CY1N0L92YC&Signature=u900lPxyDola…

horizontal_constants_icon-ch1-eps.grib2?AWSAccessKeyId=13GC1T4NT2CY1N0L92YC&Signature=eypzfw6X1QewtrLNhZvdhMk6…

In [7]:
from meteodatalab import ogd_api
import inspect

# See all fields of the Request dataclass
print(inspect.signature(ogd_api.Request))

(collection: meteodatalab.ogd_api.Collection, variable: str, reference_datetime: str, perturbed: bool, horizon: datetime.timedelta | list[datetime.timedelta]) -> None


In [ ]:
from meteodatalab import ogd_api
from datetime import timedelta
from earthkit.data import config

config.set("cache-policy", "temporary")  # avoid re-downloading

# Fetch each lead time individually — horizon does NOT support lists
das = []
hours = list(range(0, 34))  # +00h to +33h

for h in hours:
    req = ogd_api.Request(
        collection="ogd-forecasting-icon-ch1",
        variable="TOT_PREC",                    # ← underscore, not TOTPREC
        reference_datetime="latest",
        perturbed=True,
        horizon=timedelta(hours=h),             # ← single value per request
    )
    da_h = ogd_api.get_from_ogd(req)
    das.append(da_h)
    print(f"✓ +{h:02d}h")

import xarray as xr
da = xr.concat(das, dim="forecast_horizon")
da = da.assign_coords(forecast_horizon=hours)
print(da)

In [9]:
from meteodatalab import ogd_api
from datetime import timedelta


leadtimes = [timedelta(hours=h) for h in range(0, 34)]  # +00h to +33h

req = ogd_api.Request(
    collection="ogd-forecasting-icon-ch1",
    variable="TOT_PREC",
    reference_datetime="latest",   # ← was: reftime
    perturbed=True,
    horizon=leadtimes,              # ← was: leadtime
)

da = ogd_api.get_from_ogd(req)
print(da)

Earthkit-data caching is recommended. See: https://earthkit-data.readthedocs.io/en/latest/examples/cache.html


<multiple>: 0.00B [00:00, ?B/s]

horizontal_constants_icon-ch1-eps.grib2?AWSAccessKeyId=13GC1T4NT2CY1N0L92YC&Signature=eypzfw6X1QewtrLNhZvdhMk6…

<xarray.DataArray (eps: 10, ref_time: 1, lead_time: 34, cell: 1147980)> Size: 2GB
array([[[[0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         ...,
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ]]],


       [[[0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
...
         [0.        , 0.        , 0.        , ...

In [10]:
from rasterio.crs import CRS
from meteodatalab.operators import regrid

# Define ~1 km target grid over ICON-CH1-EPS domain
extent = (-0.817, 18.183, 41.183, 51.183)  # (xmin, xmax, ymin, ymax)
nx, ny = 429, 295

# Create regular lat/lon grid (EPSG:4326) and regrid
destination = regrid.RegularGrid(CRS.from_epsg(4326), nx, ny, *extent)
da_geo = regrid.iconremap(da, destination)

In [11]:
import numpy as np

# Boolean mask where member values > 0.1 mm
mask = (da_geo > 0.1)
prob_prec = mask.mean(dim="eps") * 100
prob_prec.attrs = da.attrs
prob_prec.attrs["long_name"] = "Probability of precipitation > 0.1 mm"
prob_prec.attrs["units"] = "%"

In [ ]:
from meteodatalab.operators import regrid
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from IPython.display import display

# ── 1. Regrid ─────────────────────────────────────────────────────────────────
extent = [-0.817, 18.183, 41.183, 51.183]
nx, ny = 429, 295
destination = regrid.RegularGrid(CRS.from_epsg(4326), nx, ny, *extent)
da_geo = regrid.iconremap(da, destination)

# ── 2. Probability > 0.1 mm over ensemble members ────────────────────────────
mask      = da_geo > 0.1
prob_prec = mask.mean(dim="eps") * 100   # shape: (ref_time, lead_time, y, x)

# Squeeze out ref_time (always 1) so shape becomes (lead_time, y, x)
prob_prec = prob_prec.squeeze()
print(prob_prec)   # should be (lead_time, y, x)

# ── 3. Coordinate arrays ──────────────────────────────────────────────────────
lons = np.linspace(extent[0], extent[1], nx)
lats = np.linspace(extent[2], extent[3], ny)
LON, LAT = np.meshgrid(lons, lats)

# ── 4. Time labels ────────────────────────────────────────────────────────────
lead_times   = prob_prec["lead_time"].values
n_steps      = len(lead_times)

def get_label(t):
    try:
        hours = int(np.timedelta64(t, "h") / np.timedelta64(1, "h"))
        return f"+{hours:02d}h"
    except Exception:
        return str(t)

time_labels = [get_label(t) for t in lead_times]

# ── 5. Interactive slider ─────────────────────────────────────────────────────
cmap = plt.cm.YlOrRd
norm = mcolors.Normalize(vmin=0, vmax=100)

play   = widgets.Play(value=0, min=0, max=n_steps - 1, step=1, interval=600)
slider = widgets.IntSlider(
    value=0, min=0, max=n_steps - 1, step=1,
    description="Lead time:",
    continuous_update=False,
    layout=widgets.Layout(width="80%"),
    style={"description_width": "initial"},
)
label  = widgets.Label(value=time_labels[0])
widgets.jslink((play, "value"), (slider, "value"))
output = widgets.Output()

fig, ax = plt.subplots(
    figsize=(10, 6),
    subplot_kw={"projection": ccrs.PlateCarree()}
)
cbar = None
plt.close(fig)

def update_map(step):
    global cbar
    with output:
        output.clear_output(wait=True)
        label.value = time_labels[step]

        # .isel selects the lead_time index; .values gives a 2-D (y, x) array
        data_slice = prob_prec.isel(lead_time=step).values

        ax.clear()
        ax.set_extent([extent[0], extent[1], extent[2], extent[3]])
        ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="gray")
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.add_feature(cfeature.LAND, facecolor="#f5f5f5", zorder=0)

        pcm = ax.pcolormesh(
            LON, LAT, data_slice,
            cmap=cmap, norm=norm,
            shading="auto",                     # ← fixes the flat/shape error
            transform=ccrs.PlateCarree(),
        )
        if cbar is None:
            cbar = plt.colorbar(pcm, ax=ax, label="Probability (%)", shrink=0.85)
        ax.set_title(
            f"ICON-CH1-EPS  |  P(TOT_PREC > 0.1 mm)  |  Lead {time_labels[step]}",
            fontsize=11,
        )
        ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
        display(fig)

slider.observe(lambda change: update_map(change["new"]), names="value")
display(widgets.VBox([widgets.HBox([play, slider, label]), output]))
update_map(0)

AttributeError: module 'meteodatalab.operators.regrid' has no attribute 'RegularGridCRS'